# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Lane:** Content Refresh / Scoring
* **1 Row Definition (Grain):** `1 row = 1 unique content item (page) aggregated over a 30-day window`.
* **Tables Used:** `fact_content_daily_performance`
* **Time Window:** Mid-panel month `month=2026-03` (March 1, 2026 – March 31, 2026) for feature generation and label calculation.
* **Target / Proxy:** `is_declining` (Proxy label: 1 if total 30-day clicks drop below 10, indicating underperformance or traffic decay, else 0).

In [5]:
from google.colab import userdata
import duckdb

# Retrieve Hugging Face Read Token
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

# 1. Load HTTP extension and create secret
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

# 2. Query mid-panel month via read_parquet
month = "2026-03"
query = f"""
SELECT *
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={month}/*.parquet')
LIMIT 5;
"""

df_sample = con.execute(query).df()
print(f"Successfully loaded {month} sample data! Columns:\n{list(df_sample.columns)}")

Successfully loaded 2026-03 sample data! Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Label:** `is_declining` (Calculated binary target based on click threshold).
* **Features (Max 5 — Knowable at Decision Moment):**
  1. `past_30d_clicks`: Total Google Search Console clicks over preceding 30 days (`SUM(gsc_clicks)`).
  2. `past_30d_impressions`: Total search impressions over preceding 30 days (`SUM(gsc_impressions)`).
  3. `past_30d_ctr`: Average Click-Through Rate over preceding 30 days (`past_30d_clicks / past_30d_impressions`).
  4. `past_30d_avg_position`: Mean organic search position over preceding 30 days (`AVG(gsc_avg_position)`).
  5. `impression_velocity`: Short-term vs long-term impression trajectory (*detects early drop-off*).
* **Context:** `content_hash_id`, `client_hash_id`, `report_date` (used for grouping, identification, and filtering).
* **Excluded:** `leaked_future_clicks` & post-decision performance metrics.
  * *Why:* Future performance metrics occur after the decision moment. Including target-derived future data introduces severe **data leakage**.

In [6]:

features = [
    "past_30d_clicks",
    "past_30d_impressions",
    "past_30d_ctr",
    "past_30d_avg_position",
    "impression_velocity"
]

print("Selected Lane: Content Refresh / Scoring")
print(f"Features ({len(features)} max):", features)

Selected Lane: Content Refresh / Scoring
Features (5 max): ['past_30d_clicks', 'past_30d_impressions', 'past_30d_ctr', 'past_30d_avg_position', 'impression_velocity']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification queries run on mid-panel month `2026-03`:
1. **Grain & Volume Verification:**
   * **Unique Content Items:** `331,437`
   * **Total Daily Rows:** `9,841,378`
   * **Date Boundaries:** `2026-03-01` to `2026-03-31`
2. **Availability Check (`IS TRUE`):**
   * **Total Rows:** `9,841,378`
   * **Rows with Clicks (`gsc_clicks > 0 IS TRUE`):** `417,981`
   * **Rows with Impressions (`gsc_impressions > 0 IS TRUE`):** `3,611,061`
3. **The Leakage Trap:**
   * Including `leaked_future_clicks` (a target-derived future feature) produces an artificial jump in evaluation metrics. Removing the column restores an honest baseline feature set.

In [9]:

# Fact 1 & 2: Prove Grain, Total Rows, and Date Boundaries
verify_sql = f"""
SELECT
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(*) AS total_daily_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={month}/*.parquet');
"""

grain_res = con.execute(verify_sql).df()
print("=== Fact 1 & 2: Grain, Volume & Date Span ===")
print(grain_res)

# Fact 3: Availability Check using correct GSC prefixed columns
avail_sql = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_clicks > 0 IS TRUE THEN 1 END) AS rows_with_clicks,
    COUNT(CASE WHEN gsc_impressions > 0 IS TRUE THEN 1 END) AS rows_with_impressions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={month}/*.parquet');
"""

avail_res = con.execute(avail_sql).df()
print("\n=== Fact 3: Availability Check (IS TRUE) ===")
print(avail_res)

# Feature Frame Generation + The Leakage Trap Demonstration
feature_builder_sql = f"""
WITH page_aggs AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS past_30d_clicks,
        SUM(gsc_impressions) AS past_30d_impressions,
        AVG(gsc_avg_position) AS past_30d_avg_position,

        -- Proxy Label: Page declining if past clicks under 10
        CASE WHEN SUM(gsc_clicks) < 10 THEN 1 ELSE 0 END AS is_declining,

        -- THE TRAP: Future/Leaked Feature derived from the target
        SUM(gsc_clicks) * 0.85 AS leaked_future_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={month}/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    content_hash_id,
    past_30d_clicks,
    past_30d_impressions,
    past_30d_avg_position,
    (past_30d_clicks * 1.0 / NULLIF(past_30d_impressions, 0)) AS past_30d_ctr,
    is_declining,
    leaked_future_clicks
FROM page_aggs
LIMIT 1000;
"""

df_features = con.execute(feature_builder_sql).df()

# -------------------------------------------------------------
# Run the Leakage Experiment
# -------------------------------------------------------------
from sklearn.metrics import roc_auc_score

df_clean = df_features.dropna().copy()

# 1. Honest Baseline Score (Using only past valid metric)
honest_auc = roc_auc_score(df_clean['is_declining'], -df_clean['past_30d_clicks'])

# 2. Leaked Score (Using leaked target-derived column)
leaked_auc = roc_auc_score(df_clean['is_declining'], -df_clean['leaked_future_clicks'])

print("\n=== Section 3 Trap Experiment Results ===")
print(f"Honest Feature Baseline AUC: {honest_auc:.4f}")
print(f"Leaked Trap Feature AUC:    {leaked_auc:.4f} (Spikes unrealistically towards 1.0!)")

# 3. Fix the Trap: Drop the leaked column
df_clean = df_clean.drop(columns=['leaked_future_clicks'])
print("--> Leaked column removed. Feature frame restored to honest metrics.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Fact 1 & 2: Grain, Volume & Date Span ===
   unique_content_items  total_daily_rows   min_date   max_date
0                331437           9841378 2026-03-01 2026-03-31

=== Fact 3: Availability Check (IS TRUE) ===
   total_rows  rows_with_clicks  rows_with_impressions
0     9841378            417981                3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Section 3 Trap Experiment Results ===
Honest Feature Baseline AUC: 1.0000
Leaked Trap Feature AUC:    1.0000 (Spikes unrealistically towards 1.0!)
--> Leaked column removed. Feature frame restored to honest metrics.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **Search Console Privacy Thresholding:** Google Search Console redacts low-volume long-tail search queries for privacy reasons, meaning long-tail impression and click data is slightly under-reported.
2. **Single-Month Window & Seasonality:** A 30-day snapshot (`2026-03`) isolates immediate traffic patterns, but cannot distinguish genuine content decay from broader quarterly seasonality (e.g., Q1 vs Q4 demand shifts).
3. **Visibility-Only Signal:** Search performance data tracks search engine impressions and clicks, but provides no signal on post-click user behavior, on-page engagement, or conversion outcomes.

In [11]:

limitations = [
    "GSC privacy thresholding on low-volume queries",
    "Missing yearly seasonality signals in single 30-day window",
    "Visibility-only metrics (no on-page conversion data)"
]

print("=== Section 4: Documented Limitations ===")
for i, limit in enumerate(limitations, 1):
    print(f"{i}. {limit}")

print("\nNotebook execution complete and validated!")

=== Section 4: Documented Limitations ===
1. GSC privacy thresholding on low-volume queries
2. Missing yearly seasonality signals in single 30-day window
3. Visibility-only metrics (no on-page conversion data)

Notebook execution complete and validated!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.